# Experimental bead phase diversity

Copy a completed acquisition directory to this computer, select the project
`phase_diversity_env` kernel, then edit the settings below. This notebook reads
metadata, simulates each acquired command, and displays measured/simulated
Z-stacks using the existing acquired-plane figures. Remote submission is an
explicit terminal command; no notebook cell uploads data.


In [ ]:
from pathlib import Path
import sys

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils/psf.py').exists())
sys.path.insert(0, str(root))
from experimental_10_15.bead_pd import compare_run
from IPython.display import display, Image
import json
import pandas as pd

run_dir = Path('/path/to/copied/acquisition')
output_dir = run_dir / 'analysis'
# All microscope, comparison and solver settings come from the run metadata.


In [ ]:
input_path = compare_run(run_dir, output_dir)
report = json.loads((output_dir / 'comparison.json').read_text())
display(pd.DataFrame({
    'image': report['source_indices'],
    'command': report['names'],
    'z_um': [z * 1000 for z in report['focal_z_mm']],
    'RMSE': report['rmse'],
    'relative_RMSE': report['relative_rmse'],
}))


In [ ]:
for path in sorted((output_dir / 'figures').glob('command_*.png')):
    display(Image(filename=str(path)))


## Explicit Bouchet submission

Run the following from the repository root in a terminal, replacing paths and
the login alias. Use the same preprocessing as above. `--fetch-output` watches
progress after submission and downloads the results when the job stops.

```sh
python -m experimental_10_15.submit_bead_pd submit /path/to/copied/acquisition \
  --output /path/to/submission --host mcw77@YOUR_BOUCHET_LOGIN_ALIAS \
  --fetch-output /path/to/fetched_solve
```

You can also invoke `submit`, `watch`, and `fetch` separately; see `README.md`.
Check `optimizer_success` in the downloaded result before using the coefficients.

After fetching, copy `results/correction.mat` to the microscope, set
`cfg.correctionFile` to that file in MATLAB, and run again with the same imposed
aberration and focus reference. See `WORKFLOW.md` for the short checklist.


In [ ]:
# After fetching, set this to the downloaded results directory and run this cell.
# results_dir = Path('/path/to/fetched_solve/results')
# display(pd.read_csv(results_dir / 'coefficients.csv'))
# display(json.loads((results_dir / 'result.json').read_text()))
